# Tracking with a Squared-Distance Sensor

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. An object moves along a line, and a sensor reports its squared distance to a reference point. The squared distance is a plain Julia function with no message update rules, and the precisions of the motion and of the sensor are unknown. We give the model to `fallback = NaturalGradientPropagation()`, natural-gradient message passing (Lukashchuk et al., 2026), and compare its posteriors with the exact ones, computed on a grid.

Natural-gradient propagation keeps one *site* per edge of every random variable, in the family of the variable's belief: a Gaussian for a location, a Gamma for a precision. A message with a closed form in that family is its own site. Any other message, such as a log-density pulled back through a function, has a site that moves at each iteration by a damped step towards its target, the natural gradient of the message's expected log-density under the variable's marginal. At the fixed point, each site is its target, and each marginal a stationary point of the variational free energy over its family: where the sites are Gaussian, the fixed point of conjugate-computation variational inference (Khan and Lin, 2017).

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra, StableRNGs
using Plots, PrettyTables, Printf

## The experiment

A group of researchers tracks an object moving along a one-dimensional trajectory at about one centimetre per second. They cannot see its position $z_t$. A sensor reports the squared distance $y_t$ between the object and a reference point $s$ instead, and both the servo moving the object and the sensor are noisy. The researchers model the motion as a random walk with a drift of one, and give the precisions of the motion and of the sensor, $\tau$ and $\gamma$, vague priors:

$$\begin{aligned}
 \tau &\sim \Gamma(0.01, 0.01), \qquad \gamma \sim \Gamma(0.01, 0.01),\\
 z_1 &\sim \mathcal{N}(0, \tau^{-1}), \qquad z_t \sim \mathcal{N}(z_{t - 1} + 1, \tau^{-1}),\\
 y_t &\sim \mathcal{N}\big((z_t - s)^2, \gamma^{-1}\big),
\end{aligned}$$

with $\Gamma(a, b)$ the Gamma distribution of shape $a$ and rate $b$. We simulate 50 seconds from this model, with $\tau = 1$ and a sensor noise of standard deviation 100 cm². The reference point, at 80 cm, lies beyond the end of the path, so each squared distance has one root on the object's side.

In [ ]:
nr_observations = 50
reference_point = 80
τ_true, γ_true = 1.0, 1 / 100^2

rng = StableRNG(42)
hidden_location = zeros(nr_observations)
hidden_location[1] = 1 + randn(rng) / sqrt(τ_true)
for t in 2:nr_observations
    hidden_location[t] = hidden_location[t - 1] + 1 + randn(rng) / sqrt(τ_true)
end
measurements = (hidden_location .- reference_point) .^ 2 .+ randn(rng, nr_observations) ./ sqrt(γ_true);

In [ ]:
p1 = plot(1:nr_observations, hidden_location; linewidth = 3, legend = :topleft, label = "hidden location")
hline!(p1, [reference_point]; linewidth = 3, label = "reference point")
xlabel!(p1, "time [sec]"); ylabel!(p1, "location [cm]")

p2 = scatter(1:nr_observations, measurements; label = "measurements")
plot!(p2, 1:nr_observations, (hidden_location .- reference_point) .^ 2; color = :black, label = "true squared distance")
xlabel!(p2, "time [sec]"); ylabel!(p2, "squared distance [cm²]")

plot(p1, p2; size = (1200, 500))

## The model

The squared distance is a plain function, and `μ[t] := compute_squared_distance(z[t])` makes it a deterministic node. No rule is written for it, and no approximation method is chosen for it.

In [ ]:
compute_squared_distance(z) = (z - reference_point)^2

@model function measurement_model(y)
    τ ~ Gamma(shape = 0.01, rate = 0.01)
    γ ~ Gamma(shape = 0.01, rate = 0.01)

    z[1] ~ Normal(mean = 0, precision = τ)
    μ[1] := compute_squared_distance(z[1])
    y[1] ~ Normal(mean = μ[1], precision = γ)

    for t in 2:length(y)
        z[t] ~ Normal(mean = z[t - 1] + 1, precision = τ)
        μ[t] := compute_squared_distance(z[t])
        y[t] ~ Normal(mean = μ[t], precision = γ)
    end
end

## Inference under mean field

We factorise the posterior as $q(z, \mu)\,q(\tau)\,q(\gamma)$: the path and the squared distances keep their joint, and each precision has a marginal of its own. Under this factorisation the precisions are conjugate. The `Normal` nodes send them messages of the Gamma family by their own rules, which take the expectations under $q(z, \mu)$, so each posterior is a Gamma distribution, the prior times those messages: variational message passing. The fallback works on the path:

- towards $\mu_t$, the deterministic node sends $z_t$'s message pushed forward through the squared distance, and towards $z_t$ the message on $\mu_t$ pulled back through it. Neither is a Gaussian, so each has a Gaussian site, which takes a natural-gradient step at each iteration;
- the transitions and the `Normal` observation nodes keep their exact rules, which take the Gaussian sites as they are.

A site takes one step per iteration, with the default step size `alpha = 0.5` and momentum `beta = 0.2`, so we run 300 iterations. Mean field needs a start for $q(\tau)$ and $q(\gamma)$. For the sensor, we take a start near what its noise could be, a standard deviation of about 100 cm², $\mathbb{E}[\gamma] = 10^{-4}$; the section on the start shows why.

In [ ]:
mean_field = @constraints begin
    q(z, μ, τ, γ) = q(z, μ)q(τ)q(γ)
end

initialization = @initialization begin
    q(τ) = GammaShapeRate(1.0, 1.0)
    q(γ) = GammaShapeRate(1.0, 1.0e4)
end

results = infer(
    model = measurement_model(),
    data = (y = measurements,),
    constraints = mean_field,
    initialization = initialization,
    iterations = 300,
    fallback = NaturalGradientPropagation(),
    free_energy = true,
    report = true,
    returnvars = (z = KeepLast(), τ = KeepLast(), γ = KeepLast()),
)

## What was approximated

The report lists the approximated posteriors: each $z_t$ and each $\mu_t$, with their natural-gradient sites, the messages that formed them, and whether every site had settled at the last iteration. The precisions are not among them: their posteriors are exact products of Gamma messages.

In [ ]:
@assert isempty(RxInfer.diagnostics(results.report)) # hide
@assert !RxInfer.is_approximated(results.report.approximations.variables[(:τ, nothing)]) && !RxInfer.is_approximated(results.report.approximations.variables[(:γ, nothing)]) # hide
@assert all(t -> results.report.approximations.variables[(:z, t)].sites.outcome.method === :natural_gradient, 1:nr_observations) # hide
@assert results.posteriors[:τ] isa GammaShapeRate && results.posteriors[:γ] isa GammaShapeRate # hide
results.report.approximations

## The exact posterior

The posterior has 52 dimensions, but given the two precisions the path is a hidden Markov model of one dimension. We compute it on a grid:

- **the path.** For each $y_t$ and a precision $\gamma$, the likelihood is negligible outside the points where $(z_t - s)^2$ is within nine standard deviations of $y_t$, two short intervals on either side of $s$. A grid of 100 points on each makes the transition a matrix, and the forward pass gives $\log p(y \mid \tau, \gamma)$, the backward pass the smoothing distributions of $z_t$;
- **the precisions.** On a grid of $(\log\tau, \log\gamma)$, first a coarse one to find the posterior and then a fine one of 41 × 41 points around it, the prior times $p(y \mid \tau, \gamma)$ gives the joint posterior of the precisions, and the evidence $p(y)$. The posterior of $z_t$ is the mixture of the smoothing distributions over the grid.

In [ ]:
# The points where the likelihood of yₜ is not negligible under a precision γ: (z - s)² within
# nine standard deviations of yₜ, on both sides of the reference point
function location_grid(yt, γ; points = 100, k = 9)
    lo, hi = yt - k / sqrt(γ), max(yt + k / sqrt(γ), k / sqrt(γ))
    if lo <= 0
        r = sqrt(hi); h = 2r / points
        return collect(range(reference_point - r + h / 2, reference_point + r - h / 2, length = points)), h
    end
    a, b = sqrt(lo), sqrt(hi); h = (b - a) / points
    right = collect(range(a + h / 2, b - h / 2, length = points))
    return vcat(reference_point .- reverse(right), reference_point .+ right), h
end

# The forward pass, log p(y | τ, γ), and with `smooth = true` the backward pass too: the smoothing
# means and variances of z
function hidden_markov(y, τ, γ; smooth = false)
    T = length(y)
    grids = [location_grid(y[t], γ) for t in 1:T]
    likelihood(t) = [pdf(Normal(compute_squared_distance(z), 1 / sqrt(γ)), y[t]) for z in grids[t][1]]
    transition(t) = [pdf(Normal(u + 1, 1 / sqrt(τ)), v) for u in grids[t - 1][1], v in grids[t][1]]
    α, c, K = Vector{Vector{Float64}}(undef, T), zeros(T), Vector{Matrix{Float64}}(undef, T)
    a = pdf.(Normal(0, 1 / sqrt(τ)), grids[1][1]) .* likelihood(1)
    c[1] = sum(a) * grids[1][2]; α[1] = a ./ c[1]
    for t in 2:T
        K[t] = transition(t)
        a = (K[t]' * α[t - 1]) .* grids[t - 1][2] .* likelihood(t)
        c[t] = sum(a) * grids[t][2]; α[t] = a ./ c[t]
    end
    smooth || return sum(log, c)
    β = Vector{Vector{Float64}}(undef, T); β[T] = ones(length(α[T]))
    for t in (T - 1):-1:1
        β[t] = K[t + 1] * (likelihood(t + 1) .* β[t + 1]) .* grids[t + 1][2] ./ c[t + 1]
    end
    p = [α[t] .* β[t] ./ sum(α[t] .* β[t]) for t in 1:T]
    m = [sum(p[t] .* grids[t][1]) for t in 1:T]
    return sum(log, c), m, [sum(p[t] .* (grids[t][1] .- m[t]) .^ 2) for t in 1:T]
end

prior_τ, prior_γ = Gamma(0.01, 1 / 0.01), Gamma(0.01, 1 / 0.01)

# The joint posterior of (log τ, log γ) on a grid, with the Jacobian of the logarithms
function precision_grid(y, (u_lo, u_hi), (v_lo, v_hi); n)
    us, vs = range(u_lo, u_hi, length = n), range(v_lo, v_hi, length = n)
    L = [hidden_markov(y, exp(u), exp(v)) + logpdf(prior_τ, exp(u)) + logpdf(prior_γ, exp(v)) + u + v for u in us, v in vs]
    P = exp.(L .- maximum(L))
    return (; us, vs, L, P = P ./ sum(P))
end

function exact_posterior(y)
    coarse = precision_grid(y, (log(0.05), log(20.0)), (log(1.0e-6), log(1.0e-2)); n = 25)
    mu, mv = sum(coarse.P .* coarse.us), sum(coarse.P .* coarse.vs')
    su, sv = sqrt(sum(coarse.P .* (coarse.us .- mu) .^ 2)), sqrt(sum(coarse.P .* (coarse.vs' .- mv) .^ 2))
    fine = precision_grid(y, (mu - 6su, mu + 6su), (mv - 6sv, mv + 6sv); n = 41)
    # the path, mixed over the grid points that hold the mass
    keep = findall(>(1.0e-6 * maximum(fine.P)), fine.P)
    weights = fine.P[keep] ./ sum(fine.P[keep])
    m, m2 = zeros(length(y)), zeros(length(y))
    for (w, I) in zip(weights, keep)
        _, mk, vk = hidden_markov(y, exp(fine.us[I[1]]), exp(fine.vs[I[2]]); smooth = true)
        m .+= w .* mk; m2 .+= w .* (vk .+ mk .^ 2)
    end
    logevidence = maximum(fine.L) + log(sum(exp.(fine.L .- maximum(fine.L))) * step(fine.us) * step(fine.vs))
    τs, γs = exp.(fine.us), exp.(fine.vs)
    Pτ, Pγ = vec(sum(fine.P; dims = 2)), vec(sum(fine.P; dims = 1))
    return (;
        z_mean = m, z_sd = sqrt.(m2 .- m .^ 2), logevidence,
        τ = (values = τs, density = Pτ ./ (step(fine.us) .* τs), mean = sum(Pτ .* τs), sd = sqrt(sum(Pτ .* τs .^ 2) - sum(Pτ .* τs)^2)),
        γ = (values = γs, density = Pγ ./ (step(fine.vs) .* γs), mean = sum(Pγ .* γs), sd = sqrt(sum(Pγ .* γs .^ 2) - sum(Pγ .* γs)^2)),
    )
end

exact = exact_posterior(measurements);

## Results

In [ ]:
rms(a) = sqrt(sum(abs2, a) / length(a))
summary_row(name, r) = [name mean(r.posteriors[:τ]) std(r.posteriors[:τ]) 1.0e4 * mean(r.posteriors[:γ]) 1.0e4 * std(r.posteriors[:γ]) rms(mean.(r.posteriors[:z]) - exact.z_mean) maximum(abs, std.(r.posteriors[:z]) ./ exact.z_sd .- 1) last(r.free_energy)]
exact_row = ["exact" exact.τ.mean exact.τ.sd 1.0e4 * exact.γ.mean 1.0e4 * exact.γ.sd 0.0 0.0 -exact.logevidence]
columns = ["", "E[τ]", "sd(τ)", "E[γ] × 10⁴", "sd(γ) × 10⁴", "RMS of E[z] - exact", "max |sd(z) / exact - 1|", "free energy"]
show_rows(rows) = pretty_table(rows; column_labels = columns, formatters = [fmt__printf("%.3f", 2:8)], fit_table_in_display_horizontally = false)

qτ, qγ = results.posteriors[:τ], results.posteriors[:γ]
F_mean_field = last(results.free_energy)
@printf("posterior of τ: mean %.3f, sd %.3f (exact %.3f, %.3f)\n", mean(qτ), std(qτ), exact.τ.mean, exact.τ.sd)
@printf("posterior of γ: mean %.3e, sd %.3e (exact %.3e, %.3e)\n", mean(qγ), std(qγ), exact.γ.mean, exact.γ.sd)
@assert rms(mean.(results.posteriors[:z]) - exact.z_mean) < 0.05 && maximum(abs, mean.(results.posteriors[:z]) - exact.z_mean) < 0.15 # hide
@assert maximum(abs, std.(results.posteriors[:z]) ./ exact.z_sd .- 1) < 0.14 # hide
@assert abs(mean(qτ) - exact.τ.mean) < 0.1 * exact.τ.sd && std(qτ) < 0.4 * exact.τ.sd # hide
@assert abs(mean(qγ) - exact.γ.mean) < 0.3 * exact.γ.sd && std(qγ) < 0.4 * exact.γ.sd # hide
@assert -exact.logevidence < F_mean_field < -exact.logevidence + 1.5 # hide
show_rows([summary_row("mean field", results); exact_row])

The path is accurate: the posterior means of the locations are within 0.15 cm of the exact ones, 0.04 cm in RMS, and their standard deviations within 14% of the exact ones. The plot shows the posterior of the path against the exact one; the two are hard to tell apart.

In [ ]:
m_z, s_z = mean.(results.posteriors[:z]), std.(results.posteriors[:z])
plot(1:nr_observations, exact.z_mean; ribbon = 2exact.z_sd, fillalpha = 0.3, color = :gray, label = "exact (±2 sd)", legend = :topleft, size = (900, 450))
plot!(1:nr_observations, m_z; ribbon = 2s_z, fillalpha = 0.2, color = :orange, linewidth = 2, label = "natural-gradient propagation, mean field (±2 sd)")
scatter!(1:nr_observations, hidden_location; color = :red, markersize = 3, label = "hidden location")
xlabel!("time [sec]"); ylabel!("location [cm]")

The precisions are where the factorisation shows. Both posterior means are close to the exact ones, but the posteriors are too narrow: the standard deviation of $\tau$ is about a third of the exact one, and that of $\gamma$ under 40% of it. Mean field treats the path and the precisions as independent, while a rougher path and a noisier motion explain the same measurements, and so do a smoother path and a noisier sensor. The free energy is a bound here, above $-\log p(y)$ by the divergence of the factorised posterior from the exact one.

In [ ]:
p_τ = plot(exact.τ.values, exact.τ.density; color = :black, linewidth = 2, label = "exact", xlabel = "τ", ylabel = "density", xlims = (0, 6))
plot!(p_τ, range(0.01, 6, length = 400), x -> pdf(qτ, x); color = :orange, linewidth = 2, label = "mean field")
vline!(p_τ, [τ_true]; color = :gray, linestyle = :dot, label = "true τ")
p_γ = plot(1.0e4 .* exact.γ.values, exact.γ.density ./ 1.0e4; color = :black, linewidth = 2, label = "exact", xlabel = "γ × 10⁴", ylabel = "density", xlims = (0, 4))
plot!(p_γ, range(0.001, 4, length = 400), x -> pdf(qγ, x / 1.0e4) / 1.0e4; color = :orange, linewidth = 2, label = "mean field")
vline!(p_γ, [1.0e4 * γ_true]; color = :gray, linestyle = :dot, label = "true γ")
plot(p_τ, p_γ; size = (1000, 400), bottom_margin = 5Plots.mm, left_margin = 5Plots.mm)

## The start matters under mean field

The fixed point of variational message passing under mean field depends on where it starts. Started from $q(\gamma)$ with mean 1, a sensor with a noise of 1 cm², the iterations settle in another fixed point: the path follows every measurement, the motion takes up the noise, and the free energy is higher.

In [ ]:
start_sharp = @initialization begin
    q(τ) = GammaShapeRate(1.0, 1.0)
    q(γ) = GammaShapeRate(1.0, 1.0)
end

results_sharp = infer(
    model = measurement_model(), data = (y = measurements,), constraints = mean_field, initialization = start_sharp,
    iterations = 300, fallback = NaturalGradientPropagation(), free_energy = true, returnvars = (z = KeepLast(), τ = KeepLast(), γ = KeepLast()),
)
@printf("E[γ] = %.3f, E[τ] = %.3f, free energy %.2f (from E[γ] = 10⁻⁴ at the start: %.2f)\n", mean(results_sharp.posteriors[:γ]), mean(results_sharp.posteriors[:τ]), last(results_sharp.free_energy), F_mean_field)
@assert mean(results_sharp.posteriors[:γ]) > 0.1 && mean(results_sharp.posteriors[:τ]) < 0.5 # hide
@assert last(results_sharp.free_energy) > F_mean_field + 5 # hide

The free energy tells the two apart without the exact answer: the start with a noisy sensor gives the lower one. The next section removes the factorisation between the path and the sensor's precision, and starts from the sharp sensor.

## The sensor's precision without mean field

Without mean field between the path and $\gamma$, the message of an observation towards $\gamma$ is belief propagation's: the likelihood integrated against the message of $z_t$,

$$\ell_t(\gamma) = \log \int \mathcal{N}\big(y_t \mid (z - s)^2, \gamma^{-1}\big)\, m_{z_t}(z)\, dz,$$

which is not of the Gamma family. Natural-gradient propagation gives each such message a Gamma site, which steps towards its target $\eta^\star = G(q)^{-1}\mathrm{Cov}_q[T(\gamma), \ell_t(\gamma)]$, with $T(\gamma) = (\log\gamma, \gamma)$ the Gamma's sufficient statistics and $G$ its Fisher information, by quadrature in $\log\gamma$. The posterior of $\gamma$ is the prior times the 50 sites, a Gamma distribution.

For this integral, the squared distance and the sensor's noise are one node, written with its log-density. With `μ[t] := compute_squared_distance(z[t])` in between, the integral would be against the message of $\mu_t$, whose Gaussian site can have a negative precision, and the integral is then not defined.

In [ ]:
struct SquaredDistanceSensor{Z, G} <: ContinuousUnivariateDistribution
    z::Z
    γ::G
end

BayesBase.logpdf(d::SquaredDistanceSensor, y) = logpdf(Normal(compute_squared_distance(d.z), 1 / sqrt(d.γ)), y)
BayesBase.insupport(::SquaredDistanceSensor, y) = true

@define_factor_node(node = SquaredDistanceSensor, type = Stochastic, interfaces = [:out, :z, :γ])

@model function sensor_model(y)
    τ ~ Gamma(shape = 0.01, rate = 0.01)
    γ ~ Gamma(shape = 0.01, rate = 0.01)

    z[1] ~ Normal(mean = 0, precision = τ)
    y[1] ~ SquaredDistanceSensor(z[1], γ)

    for t in 2:length(y)
        z[t] ~ Normal(mean = z[t - 1] + 1, precision = τ)
        y[t] ~ SquaredDistanceSensor(z[t], γ)
    end
end

The path and $\gamma$ now form a loop through the 50 observations, which belief propagation starts from initial messages, for $\gamma$ and for the locations. We start $\gamma$ from the sharp sensor, mean 1, that misled mean field above. The precision of the motion stays under mean field: under belief propagation, a transition's message towards a location would average over the message of its neighbour, which can be an improper Gaussian, and that expectation is not defined. With 50 Gamma sites and 50 Gaussian ones, we run 300 iterations again.

In [ ]:
path_and_sensor = @constraints begin
    q(z, τ, γ) = q(z, γ)q(τ)
end

initialization_bp = @initialization begin
    q(τ) = GammaShapeRate(1.0, 1.0)
    μ(γ) = GammaShapeRate(1.0, 1.0)
    μ(z) = NormalMeanVariance(0.0, 100.0)
end

results_bp = infer(
    model = sensor_model(),
    data = (y = measurements,),
    constraints = path_and_sensor,
    initialization = initialization_bp,
    iterations = 300,
    fallback = NaturalGradientPropagation(),
    free_energy = true,
    report = true,
    returnvars = (z = KeepLast(), τ = KeepLast(), γ = KeepLast()),
)

The report now lists $\gamma$ among the approximated posteriors, with its 50 Gamma sites, all settled. It also warns about the messages towards the locations: with $\gamma$'s message a Gamma distribution, the fallback cannot integrate the likelihood against it, and averages the log-likelihood over it instead, as variational message passing does.

In [ ]:
records_bp = results_bp.report.approximations
@assert records_bp.variables[(:γ, nothing)].sites.outcome.method === :natural_gradient # hide
@assert [d.kind for d in RxInfer.diagnostics(results_bp.report)] == [:expectation_over_messages] # hide
@assert results_bp.posteriors[:γ] isa GammaShapeRate # hide
records_bp

In [ ]:
qγ_bp, qτ_bp = results_bp.posteriors[:γ], results_bp.posteriors[:τ]
F_bp = last(results_bp.free_energy)
@assert abs(mean(qγ_bp) - exact.γ.mean) < abs(mean(qγ) - exact.γ.mean) # hide
@assert 0.6 * exact.γ.sd < std(qγ_bp) < 0.7 * exact.γ.sd && std(qγ_bp) > 1.6 * std(qγ) # hide
@assert abs(mean(qτ_bp) - exact.τ.mean) > abs(mean(qτ) - exact.τ.mean) && std(qτ_bp) < 0.4 * exact.τ.sd # hide
@assert rms(mean.(results_bp.posteriors[:z]) - exact.z_mean) < 0.05 # hide
@assert F_bp < -exact.logevidence # hide
show_rows([summary_row("mean field", results); summary_row("q(z, γ) q(τ)", results_bp); exact_row])

From the sharp start, the iterations find the noisy sensor. With the path and the sensor's precision together, the posterior of $\gamma$ is closer to the exact one in both mean and spread: its standard deviation is about two thirds of the exact one, against under 40% under mean field. The precision of the motion, still under mean field, is as narrow as before and further from the exact mean. The path is as accurate as before.

The free energy is now below $-\log p(y)$. With a loop through the observations it is Bethe's, with the average energies of the nodes without rules: a diagnostic of the fixed point, not a bound, so it does not rank this run against the mean-field one.

In [ ]:
p_γ_bp = plot(1.0e4 .* exact.γ.values, exact.γ.density ./ 1.0e4; color = :black, linewidth = 2, label = "exact", xlabel = "γ × 10⁴", ylabel = "density", xlims = (0, 4))
plot!(p_γ_bp, range(0.001, 4, length = 400), x -> pdf(qγ, x / 1.0e4) / 1.0e4; color = :orange, linewidth = 2, label = "mean field")
plot!(p_γ_bp, range(0.001, 4, length = 400), x -> pdf(qγ_bp, x / 1.0e4) / 1.0e4; color = :blue, linewidth = 2, label = "q(z, γ) q(τ)")
vline!(p_γ_bp, [1.0e4 * γ_true]; color = :gray, linestyle = :dot, label = "true γ")
plot!(p_γ_bp; size = (700, 400), bottom_margin = 5Plots.mm, left_margin = 5Plots.mm)

## Summary

- **One keyword.** The squared distance is a plain Julia function, and `fallback = NaturalGradientPropagation()` runs the model without a rule, an approximation method or a form constraint for it. The report says what was approximated: the path, by natural-gradient sites.
- **What is conjugate.** Under mean field, both precisions have Gamma messages from the `Normal` nodes' own rules and exact Gamma posteriors; the sites work on the path alone. Without mean field between the path and $\gamma$, $\gamma$'s messages are belief propagation's, not of the Gamma family, and its posterior is the prior times 50 Gamma sites.
- **Accuracy.** The path matches the exact posterior to a few hundredths of a centimetre. Mean field makes both precisions too narrow; keeping $\gamma$ with the path widens its posterior to two thirds of the exact spread.

## References

- M. Lukashchuk, K. Yemets, A. Ledbetter and İ. Şenöz (2026). Information geometry of message passing. arXiv:2608.15922.
- M. E. Khan and W. Lin (2017). Conjugate-computation variational inference: converting variational inference in non-conjugate models to inferences in conjugate models. *AISTATS*.